# Steepest Descent — Interactive Numerical Experiments

Explore exact-line-search **Steepest Descent** on real symmetric
positive-definite (SPD) and complex Hermitian positive-definite (HPD)
systems. Compare convergence with Conjugate Gradient, examine conditioning,
and measure the true residual and relative solution error.

All matrices use a fixed random seed and a known exact solution for
reproducibility. Select the controls and click **Run experiment**.


In [ ]:
from pathlib import Path
import sys
import time

# Binder or local Jupyter: find the repository root.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "gradient_methods").is_dir() and (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.sparse.linalg import cg as scipy_cg

from gradient_methods.steepest_descent_solver import solve_steepest_descent
from krylov_methods.conjugate_gradient_solver import solve_conjugate_gradient
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, residual_plot, error_plot, report,
)


## 1. Mathematical model

For $A$ real SPD (or complex HPD), solve $Ax=b$ by minimizing the energy

<p align="center">$\displaystyle f(x)=\frac12 x^HAx-\operatorname{Re}(b^Hx).$</p>

With residual $r_k=b-Ax_k$, Steepest Descent uses the negative gradient
direction and exact line search:

<p align="center">$\displaystyle
\alpha_k=\frac{r_k^Hr_k}{r_k^HAr_k},\qquad
x_{k+1}=x_k+\alpha_k r_k.
$</p>

Here $x^H$ denotes conjugate transpose; for real systems it is simply $x^T$.
Steepest Descent minimizes the energy **along the current residual direction**.
CG builds conjugate directions and generally converges in fewer iterations.

The experiments use $A=Q\Lambda Q^H$ with positive, geometrically spaced
eigenvalues. The condition number is

<p align="center">$\displaystyle \kappa_2(A)=\lambda_{\max}/\lambda_{\min}.$</p>

We generate $x^*$ and set $b=Ax^*$, allowing comparison of
the **true residual** $\|b-Ax_k\|_2$ and **relative solution error**
$\|x_k-x^*\|_2/\|x^*\|_2$.


## 2. Interactive comparison with Conjugate Gradient

Adjust the system and stopping tolerance. The plots compare:

- Steepest Descent from `gradient_methods.steepest_descent_solver`.
- Your Conjugate Gradient implementation from `krylov_methods`.
- SciPy CG as an independent reference.

All solvers start at $x_0=0$ and use the same absolute/relative stopping rule.
Reported times are solver-call measurements on small educational matrices;
they are not rigorous benchmarks.


In [ ]:
kind = widgets.Dropdown(
    options=[("Real SPD", "real"), ("Complex Hermitian PD", "complex")],
    value="real", description="System",
)
size = widgets.IntSlider(value=20, min=5, max=80, step=5, description="Size n")
condition = widgets.FloatLogSlider(value=1e2, base=10, min=1, max=5,
                                   step=0.5, description="Cond(A)")
tol = widgets.FloatLogSlider(value=1e-8, base=10, min=-11, max=-5,
                             step=1, description="rtol")
budget = widgets.IntSlider(value=2000, min=100, max=8000, step=100,
                           description="Max iterations")
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        A, b, x_true = (
            spd_problem(size.value, condition.value) if kind.value == "real"
            else hermitian_pd_problem(size.value, condition.value)
        )
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvalsh(A)
        print("SYSTEM INFORMATION")
        system_label = {"real": "Real SPD", "complex": "Complex Hermitian PD"}[kind.value]
        print(f"Matrix: {system_label}, n={size.value}, dtype={A.dtype}")
        print(f"||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Eigenvalue interval=[{eigs[0]:.3e}, {eigs[-1]:.3e}]")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}, ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"True residual threshold={threshold:.3e}, budget={budget.value}\n")

        start = time.perf_counter()
        sd = solve_steepest_descent(A, b, rtol=tol.value, atol=1e-12,
                                    max_iter=budget.value, store_history=True)
        sd_time = time.perf_counter() - start
        start = time.perf_counter()
        cg = solve_conjugate_gradient(A, b, rtol=tol.value, atol=1e-12,
                                      max_iter=budget.value, store_history=True)
        cg_time = time.perf_counter() - start

        scipy_hist = [np.linalg.norm(b)]
        scipy_error = [1.0]
        def callback(xk):
            scipy_hist.append(np.linalg.norm(b-A@xk))
            scipy_error.append(np.linalg.norm(xk-x_true)/np.linalg.norm(x_true))
        start = time.perf_counter()
        x_scipy, info = scipy_cg(A, b, rtol=tol.value, atol=1e-12,
                                 maxiter=budget.value, callback=callback)
        scipy_time = time.perf_counter() - start

        report("Steepest Descent", sd.solution, x_true, b, A, sd.iterations, sd.converged)
        report("Our CG", cg.solution, x_true, b, A, cg.iterations, cg.converged)
        report("SciPy CG", x_scipy, x_true, b, A, len(scipy_hist)-1, info == 0)
        print(f"Times (s): SD={sd_time:.4f}, our CG={cg_time:.4f}, SciPy CG={scipy_time:.4f}")
        print(f"SD status: {sd.message}; CG status: {cg.message}; SciPy info={info}")

        histories = {
            "Steepest Descent": sd.residual_norms,
            "Our CG": cg.residual_norms,
            "SciPy CG": scipy_hist,
        }
        error_histories = {
            "Steepest Descent": [
                np.linalg.norm(xk-x_true)/np.linalg.norm(x_true) for xk in sd.history
            ],
            "Our CG": [
                np.linalg.norm(xk-x_true)/np.linalg.norm(x_true) for xk in cg.history
            ],
            "SciPy CG": scipy_error,
        }
        residual_plot(histories, threshold, "Steepest Descent vs CG — true residual")
        error_plot(error_histories, "Steepest Descent vs CG — relative solution error")
        print("Note: SD decreases the quadratic energy with exact line search, "
              "but the Euclidean residual norm is not guaranteed to decrease each step.")


button.on_click(run)
display(widgets.VBox([kind, size, condition, tol, budget, button, output]))


## 3. Numerical exploration — Why conditioning matters

Even when Steepest Descent uses the optimal step along each residual, its
iterations can zig-zag in the geometry of a strongly elongated quadratic.
The next experiment uses a **two-dimensional** SPD matrix
$A=Q\operatorname{diag}(1,\kappa)Q^T$ and displays energy-level contours and
the first few Steepest Descent iterates. Compare the path with CG.

For large $\kappa$, the iteration may make slow progress even when every
individual line search is exact. The experiment is independent of the
larger-system comparison above.


In [ ]:
path_kappa = widgets.FloatLogSlider(value=1e2, base=10, min=0, max=4,
                                     step=0.5, description="2D Cond(A)")
path_steps = widgets.IntSlider(value=12, min=2, max=40, step=2,
                                description="SD steps")
path_button = widgets.Button(description="Explore optimization path", button_style="info")
path_output = widgets.Output()


def run_path(_):
    with path_output:
        clear_output(wait=True)
        angle = np.pi / 5
        Q = np.array([[np.cos(angle), -np.sin(angle)],
                      [np.sin(angle), np.cos(angle)]])
        A = Q @ np.diag([1.0, path_kappa.value]) @ Q.T
        x_true = np.array([1.0, 1.0])
        b = A @ x_true
        sd = solve_steepest_descent(A, b, rtol=0.0, atol=0.0,
                                    max_iter=path_steps.value, store_history=True)
        cg = solve_conjugate_gradient(A, b, rtol=1e-14, atol=1e-14,
                                      max_iter=2, store_history=True)
        sd_path = sd.history
        cg_path = cg.history

        xgrid = np.linspace(min(sd_path[:, 0].min(), 0)-0.4, 1.5, 160)
        ygrid = np.linspace(min(sd_path[:, 1].min(), 0)-0.4, 1.5, 160)
        X, Y = np.meshgrid(xgrid, ygrid)
        dx, dy = X-x_true[0], Y-x_true[1]
        energy = 0.5*(A[0,0]*dx**2 + 2*A[0,1]*dx*dy + A[1,1]*dy**2)
        values = energy[energy > 0]
        low = max(np.min(values), 1e-6)
        high = max(np.max(energy), low*10)
        fig, ax = plt.subplots(figsize=(7, 5.5))
        ax.contour(X, Y, energy, levels=np.geomspace(low, high, 14),
                   colors="0.65", linewidths=0.8)
        ax.plot(sd_path[:,0], sd_path[:,1], "o-", ms=3,
                label="Steepest Descent")
        ax.plot(cg_path[:,0], cg_path[:,1], "s--", ms=5, label="CG")
        ax.scatter([x_true[0]], [x_true[1]], marker="*", s=150,
                   color="black", label="Exact minimizer")
        ax.set(xlabel="$x_1$", ylabel="$x_2$",
               title=f"Optimization path — condition number {path_kappa.value:.1e}")
        ax.legend()
        ax.grid(alpha=0.2)
        fig.tight_layout()
        plt.show()
        print(f"Steepest Descent iterations={sd.iterations}, "
              f"relative error={np.linalg.norm(sd.solution-x_true)/np.linalg.norm(x_true):.3e}")
        print(f"CG iterations={cg.iterations}, "
              f"relative error={np.linalg.norm(cg.solution-x_true)/np.linalg.norm(x_true):.3e}")


path_button.on_click(run_path)
display(widgets.VBox([path_kappa, path_steps, path_button, path_output]))
